In [3]:
import pandas as pd
import io
import requests

START_YEAR = 2013
END_YEAR = 2023

# Representative coordinate in Jalisco (near Guadalajara)
LAT = 20.6736
LON = -103.3440

# 1) AGRICULTURE DATA (Datamx / SIAP)
AGRI_URL = "https://datamx.io/dataset/produccion-agricola-en-mexico-1980-2024/resource/8d22bea7-25d6-44d2-bea3-53fd4b947024/download/data.csv"
# Source: Producción agrícola en México (1980-2024), from SIAP cierre agrícola.

# 2) HOMICIDE DATA (Figshare, INEGI/SESNSP-based) -> you must paste real file URL
HOMICIDE_URL = "PASTE_FIGSHARE_DOWNLOAD_URL_HERE"

# 3) CLIMATE DATA (NASA POWER)
NASA_URL = (
    "https://power.larc.nasa.gov/api/temporal/monthly/point"
    f"?parameters=T2M,PRECTOT&community=AG&longitude={LON}&latitude={LAT}"
    f"&start={START_YEAR}&end={END_YEAR}&format=CSV"
)


# ---------- helper: safe column resolver ----------

def resolve_col(cols, *candidates):
    cols_up = {c.upper(): c for c in cols}
    for cand in candidates:
        if cand in cols_up:
            return cols_up[cand]
    raise KeyError(f"None of {candidates} found. Available columns: {list(cols)}")


# ---------- 1. LOAD & FILTER AGRI DATA ----------

print("Downloading agricultural data...")
agri_resp = requests.get(AGRI_URL)
agri_resp.raise_for_status()
agri = pd.read_csv(io.StringIO(agri_resp.text))

# Normalize header
agri.columns = [c.strip().upper() for c in agri.columns]

# Resolve key columns
year_col = resolve_col(agri.columns, "AÑO", "ANIO", "ANO", "YEAR")
ent_col = resolve_col(agri.columns, "ENTIDAD", "NOM_ENT")
cultivo_col = resolve_col(agri.columns, "CULTIVO")
rend_col = resolve_col(agri.columns, "RENDIMIENTO")

# Make state names uppercase for filtering
agri[ent_col] = agri[ent_col].astype(str).str.upper()

# Filter Jalisco & year range
agri_jal = agri[
    (agri[ent_col] == "JALISCO")
    & (agri[year_col].between(START_YEAR, END_YEAR))
].copy()

if agri_jal.empty:
    raise ValueError("No Jalisco rows found in agri data for selected years. Check source / filters.")

def compute_yield(df, pattern):
    mask = df[cultivo_col].astype(str).str.contains(pattern, case=False, na=False)
    sub = df[mask].copy()
    if sub.empty:
        return pd.DataFrame(columns=["Year", "yield_t_ha"])

    # Prefer area-weighted avg if superficie cosechada exists
    surf_cols = [c for c in sub.columns if "SUPERFICIE_COSECHADA" in c]
    if surf_cols:
        surf_col = surf_cols[0]
        sub = sub[sub[rend_col].notna() & sub[surf_col].notna()]
        if sub.empty:
            return pd.DataFrame(columns=["Year", "yield_t_ha"])
        g = (
            sub.groupby(year_col)
               .apply(lambda g: (g[rend_col] * g[surf_col]).sum() / g[surf_col].sum())
               .rename("yield_t_ha")
               .reset_index()
        )
    else:
        # simple mean if no area column
        g = (
            sub[sub[rend_col].notna()]
            .groupby(year_col)[rend_col]
            .mean()
            .rename("yield_t_ha")
            .reset_index()
        )
    g = g.rename(columns={year_col: "Year"})
    return g

maize = compute_yield(agri_jal, r"MA[IÍ]Z")
beans = compute_yield(agri_jal, r"FRIJOL")
sorghum = compute_yield(agri_jal, r"SORGO")

# Base frame with all years
df = pd.DataFrame({"Year": range(START_YEAR, END_YEAR + 1)})

if not maize.empty:
    df = df.merge(maize, on="Year", how="left")
    df = df.rename(columns={"yield_t_ha": "Maize_Yield_t_ha"})
else:
    df["Maize_Yield_t_ha"] = pd.NA

if not beans.empty:
    df = df.merge(beans, on="Year", how="left")
    df = df.rename(columns={"yield_t_ha": "Bean_Yield_t_ha"})
else:
    df["Bean_Yield_t_ha"] = pd.NA

if not sorghum.empty:
    df = df.merge(sorghum, on="Year", how="left")
    df = df.rename(columns={"yield_t_ha": "Sorghum_Yield_t_ha"})
else:
    df["Sorghum_Yield_t_ha"] = pd.NA


# ---------- 2. HOMICIDE RATES (YOU MUST SET URL) ----------

if "PASTE_FIGSHARE_DOWNLOAD_URL_HERE" in HOMICIDE_URL:
    print("\n[!] Skipping homicide merge: set HOMICIDE_URL to the actual Figshare CSV download URL.")
    hom = pd.DataFrame(columns=["Year", "Homicide_Rate_per_100k"])
else:
    print("Downloading homicide data...")
    hom_resp = requests.get(HOMICIDE_URL)
    hom_resp.raise_for_status()
    hom = pd.read_csv(io.StringIO(hom_resp.text))
    hom.columns = [c.strip().upper() for c in hom.columns]

    state_col = resolve_col(hom.columns, "STATE", "ESTADO", "ENTIDAD")
    year_h_col = resolve_col(hom.columns, "YEAR", "AÑO", "ANIO", "ANO")
    rate_col = resolve_col(hom.columns, "HOMICIDE_RATE_PER_100K", "TASA", "RATE", "TASA_HOMICIDIOS")

    hom[state_col] = hom[state_col].astype(str).str.upper()
    jal_hom = hom[
        (hom[state_col] == "JALISCO")
        & (hom[year_h_col].between(START_YEAR, END_YEAR))
    ][[year_h_col, rate_col]].rename(
        columns={year_h_col: "Year", rate_col: "Homicide_Rate_per_100k"}
    )

    df = df.merge(jal_hom, on="Year", how="left")


# ---------- 3. CLIMATE (NASA POWER) ----------

print("Downloading NASA POWER data...")
clim_resp = requests.get(NASA_URL)
clim_resp.raise_for_status()
lines = clim_resp.text.splitlines()

# Find header row that starts with YEAR or YEAR,MO
start_idx = None
for i, line in enumerate(lines):
    if line.startswith("YEAR"):
        start_idx = i
        break

if start_idx is None:
    print("[!] Could not automatically detect data section in NASA POWER response.")
    annual = pd.DataFrame(columns=["Year", "Avg_Temperature_C", "Annual_Rainfall_mm"])
else:
    clim = pd.read_csv(io.StringIO("\n".join(lines[start_idx:])))
    # Expect columns: YEAR, MO, T2M, PRECTOT etc.
    if "MO" in clim.columns:
        annual = (
            clim.groupby("YEAR")
                .agg(
                    Avg_Temperature_C=("T2M", "mean"),
                    Annual_Rainfall_mm=("PRECTOT", "sum"),
                )
                .reset_index()
                .rename(columns={"YEAR": "Year"})
        )
    else:
        # If already annual in response
        clim = clim.rename(columns={"YEAR": "Year"})
        annual = clim[["Year", "T2M", "PRECTOT"]].rename(
            columns={"T2M": "Avg_Temperature_C", "PRECTOT": "Annual_Rainfall_mm"}
        )

    df = df.merge(annual, on="Year", how="left")


# ---------- 4. ADD SOURCES & SAVE ----------

df["Crop_Data_Source"] = "SIAP cierre agrícola via Datamx.io (Producción agrícola en México 1980-2024)"
df["Climate_Data_Source"] = "NASA POWER (T2M, PRECTOT), monthly aggregated to annual for Jalisco coordinate"
df["Crime_Data_Source"] = (
    "INEGI/SESNSP via Figshare homicide rates dataset"
    if "Homicide_Rate_per_100k" in df.columns
    else "NOT_MERGED_SET_URL"
)

out_path = "jalisco_food_institution_climate_2013_2023.csv"
df.to_csv(out_path, index=False)

print("\nSaved:", out_path)
print(df)


KeyError: "None of ('AÑO', 'ANIO', 'ANO', 'YEAR') found. Available columns: ['AÃ\\x91O', 'CVE_ENT', 'ENTIDAD', 'CULTIVO', 'VARIEDAD', 'UNIDAD_MEDIDA', 'TIPO_TECNOLOGIA', 'TIPO_PRODUCCION', 'TIPO_MERCADO', 'SUPERFICIE_SEMBRADA', 'SUPERFICIE_COSECHADA', 'SUPERFICIE_SINIESTRADA', 'VOLUMEN_PRODUCCION', 'RENDIMIENTO', 'PRECIO_MEDIO_RURAL', 'VALOR_PRODUCCION']"